# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sorgerator/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Optimization Flags and Content Performance
**The Claim:** The paper states that content with 2 to 3 internal optimization flags actually scores higher in overall "Health" than content with zero flags. This happens because many diagnostic flags only trigger on pages that already generate enough impressions or behavioral data to be measured.

**Methodology Question:** *Does the validation design properly isolate "optimization flags" as an independent variable, or is the flag assignment mechanism inherently confounded by a minimum traffic threshold? Given that zero-visibility pages cannot receive tactical flags, how does the methodology account for the bias that this finding might simply be demonstrating that "pages with measurable traffic perform better than pages with zero traffic"?*

### Finding 2: AI-Generated Content Penalties
**The Claim:** The research concludes that AI-generated content is not penalized by default, arguing that content quality and editorial processes drive performance outcomes rather than the mere presence of AI assistance.

**Methodology Question:** *Where does the exact origin label (e.g., "OpenAI" versus "Gemini") come from across the analyzed pieces, and how does the study quantify the degree of human intervention? Furthermore, does the dataset's reliance on a "mostly AI-authored portfolio" lack a statistically robust, strictly human-authored control group, and if so, does this absence weaken the validation of the claim that there is no blanket AI penalty?*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Load Data & Recreate Week 5 Variables
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Rebuild the target and feature columns exactly as you had them
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
target_col = 'is_declining_label'
group_col = 'client_id'
leakage_cols = ['content_id', 'client_id', 'trend_direction', 'trend_pct', target_col]

numeric_cols = df.select_dtypes(include=[np.number, bool]).columns
feature_cols = [c for c in numeric_cols if c not in leakage_cols]

X = df[feature_cols]
y = df[target_col]
groups = df[group_col]

# 2. Calculate the Naive Base Rate (Majority Class)
base_rate = y.mean() if y.mean() > 0.5 else 1 - y.mean()
print(f"--- BASE RATE (Naive Guessing) ---")
print(f"Base Rate: {base_rate:.4f}\n")

# 3. Your Week 5 Random Forest Pipeline
rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('model', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5))
])

# ==========================================
# BEFORE: The Naive Random Split
# ==========================================
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_pipeline.fit(X_train_rand, y_train_rand)
preds_rand = rf_pipeline.predict(X_test_rand)

rand_accuracy = accuracy_score(y_test_rand, preds_rand)
print("--- BEFORE: Random Split ---")
print(f"Accuracy:  {rand_accuracy:.4f}")
print(f"True Skill (Accuracy - Base Rate): {rand_accuracy - base_rate:.4f}\n")

# ==========================================
# AFTER: The Honest Grouped Split (From Week 5)
# ==========================================
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_pipeline.fit(X_train_grp, y_train_grp)
preds_grp = rf_pipeline.predict(X_test_grp)

grp_accuracy = accuracy_score(y_test_grp, preds_grp)
print("--- AFTER: Honest Split (Grouped by client_id) ---")
print(f"Accuracy:  {grp_accuracy:.4f}")
print(f"True Skill (Accuracy - Base Rate): {grp_accuracy - base_rate:.4f}\n")

# The Gap
print(f"--- THE GAP ---")
print(f"Performance lost when forced to generalize: {rand_accuracy - grp_accuracy:.4f}")

--- BASE RATE (Naive Guessing) ---
Base Rate: 0.5421

--- BEFORE: Random Split ---
Accuracy:  0.7427
True Skill (Accuracy - Base Rate): 0.2006

--- AFTER: Honest Split (Grouped by client_id) ---
Accuracy:  0.6688
True Skill (Accuracy - Base Rate): 0.1268

--- THE GAP ---
Performance lost when forced to generalize: 0.0738


**The Gap:** 7.4%

**Interpretation:** When forced to generalize to unseen clients via the grouped split, the model lost over a third of its "true skill" (dropping from 0.2006 to 0.1268 above the base rate). This confirms that the naive random split allowed the model to cheat by memorizing client-specific baseline traffic patterns. The grouped split provides an honest, measured look at how the model will actually perform on new, unseen accounts.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# 1. Feature Importance Check 
rf_model = rf_pipeline.named_steps['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

print("--- Top 5 Feature Importances ---")
print(importances.head(5))

# 2. Window & Sibling Verification
print("\n--- Leakage Verification ---")
leaky_cols = ['trend_pct', 'trend_direction', 'is_declining_label']
found_leaks = [c for c in leaky_cols if c in feature_cols]
print(f"Leaky columns found in features: {found_leaks}")

--- Top 5 Feature Importances ---
impressions_prev_30d     0.341202
impressions_90d          0.105458
days_with_impressions    0.082218
avg_position             0.079214
impressions_last_30d     0.067612
dtype: float64

--- Leakage Verification ---
Leaky columns found in features: []


**Leakage Audit Findings:**
1. **Label-Derived Features:** The highest feature importance sits at around ~0.34 (`impressions_prev_30d`). No single feature dominates with a near-1.0 importance, and we explicitly verified that direct label leaks (`trend_direction`, `trend_pct`) are absent from the feature set.
2. **Future/Overlapping Windows:** All features used represent historical aggregates (`_90d`, `_last_30d`) that are strictly knowable at the moment of prediction. The timeline is clean.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original Claim (Too Bold):** 
"The Random Forest model accurately predicts which content will decline next month and should be used to automate our content refresh pipeline."

**Rewritten Claim (Safe Language):**
"The Random Forest model provides a **directional** signal for content decay. We **observed** a **measured** True Skill of 0.1268 over the base rate when generalizing to unseen clients, indicating it can serve as a robust **decision-support** tool for prioritizing content refresh queues."

## Self-check

Before you submit, confirm each line honestly:

- [ **X** ] Every section above is filled — markdown thinking AND the code that backs it
- [ **X** ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ **X** ] No client names, URLs, or private queries anywhere
- [ **X** ] My claims use careful words: observed, measured, directional, decision-support
- [ **X** ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.